In [ ]:
import pandas as pd
import numpy as np
from tools import sherlock

In [ ]:
df = pd.read_excel(r"..\data\raw\consultations_janvier2026.xlsx", sheet_name=0, skiprows=1, nrows=20)

# Exploration

In [ ]:
sherlock(df)

In [ ]:
df = df.rename(str.lower, axis='columns')
df.columns = df.columns.str.strip().str.replace(' ', '_')
df['taux_de_réservation'] = df['taux_de_réservation'].str.strip().str.replace('%', '').astype(float)

In [ ]:
df['period_label'] = df['mois']
mois_fr = {
    "Janvier": 1, "Février": 2, "Mars": 3, "Avril": 4,
    "Mai": 5, "Juin": 6, "Juillet": 7, "Août": 8,
    "Septembre": 9, "Octobre": 10, "Novembre": 11, "Décembre": 12
}
df['mois_num'] = df['mois'].str.split().str[0].map(mois_fr).astype(str)
df['année'] = df['mois'].str.split().str[1]
df['mois'] = df['mois_num'] + " " + df['année']
df['mois'] = pd.to_datetime(df['mois'], format="%m %Y").dt.to_period('M')
df['année'] = df['année'].astype(int)
df = df.drop(columns=['mois_num'])

In [ ]:
sherlock(df)

In [ ]:
df

# Feature

Consultations : variation en % entre le mois M et le mois M-1 + Y et Y-1

In [ ]:
df['var_consultations_pct'] = (df['consultations'] - df['consultations'].shift(1)) / df['consultations'].shift(1) * 100
df['var_consultations_pct_annuel'] = (df['consultations'] - df['consultations'].shift(12)) / df['consultations'].shift(12) * 100

Réservation & tx de résa : variation en % et point entre Y et Y-1

In [ ]:
df['var_resa_pct_annuel'] = (df['réservations'] - df['réservations'].shift(12)) / df['réservations'].shift(12) * 100
df['var_resa_pts_annuel'] = (df['taux_de_réservation'] - df['taux_de_réservation'].shift(12))
df = df.replace(np.inf, np.nan)
df

### Export

In [ ]:
df.to_csv(r"..\data\processed\airbnb.csv", index=False, sep=';', decimal=',', encoding='utf-8-sig')